# 🗑️ SmartWaste AI — Phase 2: EDA, Preprocessing & Dataset

This notebook covers:
1. **EDA** — class distribution, sample images, dimensions, channel stats
2. **Preprocessing** — train vs val/test transform pipelines (explained)
3. **WasteDataset** — custom `torch.utils.data.Dataset` walkthrough
4. **DataLoaders** — train/val/test with batch inspection
5. **Validation** — 10-point automated check

**Prerequisites:** Phase 1 must be complete (dataset in `dataset/processed/`)


In [ ]:
import sys, os
PROJECT_ROOT = '/content/SmartWaste_Ai'   # Change if running locally
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')


## 1. Exploratory Data Analysis


In [ ]:
from ml.eda import count_images, plot_class_distribution, write_imbalance_report
from ml.config import cfg

counts = count_images()

# Print summary table
CLASSES = list(cfg.CLASS_NAMES)
print(f'\n{"Class":<14} {"Train":>6} {"Val":>6} {"Test":>6} {"Total":>7}')
print('─' * 50)
for cls in CLASSES:
    train_n = counts['train'].get(cls, 0)
    val_n   = counts['val'].get(cls, 0)
    test_n  = counts['test'].get(cls, 0)
    total   = train_n + val_n + test_n
    print(f'  {cls:<12} {train_n:>6} {val_n:>6} {test_n:>6} {total:>7}')
print('─' * 50)
grand = sum(sum(counts[s][c] for s in counts) for c in CLASSES)
print(f'  {"TOTAL":<12} {sum(counts["train"].values()):>6} '
      f'{sum(counts["val"].values()):>6} {sum(counts["test"].values()):>6} {grand:>7}')


In [ ]:
from IPython.display import Image as IPImage, display
from ml.eda import plot_class_distribution

out = plot_class_distribution(counts)
display(IPImage(str(out)))


In [ ]:
from ml.eda import plot_sample_grid

out = plot_sample_grid(samples_per_class=4)
display(IPImage(str(out)))


In [ ]:
# Image dimension analysis (scans up to 100 images per class)
from ml.eda import plot_image_dimensions
out = plot_image_dimensions(max_per_class=100)
display(IPImage(str(out)))


In [ ]:
from ml.eda import plot_channel_statistics
out = plot_channel_statistics(max_per_class=50)
display(IPImage(str(out)))


In [ ]:
from ml.eda import plot_augmentation_preview
out = plot_augmentation_preview(n_augmented=5)
display(IPImage(str(out)))


In [ ]:
from ml.eda import write_imbalance_report
write_imbalance_report(counts)


## 2. Preprocessing Transforms — Explained

We use **different pipelines** for training vs validation/testing:

| Transform | Train | Val/Test | Reason |
|-----------|-------|----------|--------|
| RandomResizedCrop(224) | ✅ | ❌ | Random crop = data augmentation |
| Resize(256) + CenterCrop(224) | ❌ | ✅ | Deterministic, consistent |
| RandomHorizontalFlip | ✅ | ❌ | Doubles effective dataset |
| RandomRotation(15°) | ✅ | ❌ | Rotational invariance |
| ColorJitter | ✅ | ❌ | Lighting robustness |
| Normalize(ImageNet stats) | ✅ | ✅ | Mandatory for ResNet18 |


In [ ]:
from ml.preprocessing import get_train_transforms, get_val_transforms

print('TRAINING TRANSFORMS:')
print(get_train_transforms())

print('\nVALIDATION TRANSFORMS:')
print(get_val_transforms())


## 3. WasteDataset — Custom PyTorch Dataset


In [ ]:
from ml.dataset import WasteDataset
from ml.preprocessing import get_train_transforms, get_val_transforms, get_test_transforms

# Create all three splits
train_ds = WasteDataset('train', transform=get_train_transforms(), verbose=True)
val_ds   = WasteDataset('val',   transform=get_val_transforms(),   verbose=True)
test_ds  = WasteDataset('test',  transform=get_test_transforms(),  verbose=True)

print(f'\nTrain dataset: {train_ds}')
print(f'Val   dataset: {val_ds}')
print(f'Test  dataset: {test_ds}')

In [ ]:
import torch

# Demonstrate __getitem__
img_tensor, label = train_ds[0]

print(f'Image tensor shape : {img_tensor.shape}')    # [3, 224, 224]
print(f'Image tensor dtype : {img_tensor.dtype}')    # float32
print(f'Image tensor min   : {img_tensor.min():.4f}') # Negative after normalisation
print(f'Image tensor max   : {img_tensor.max():.4f}')
print(f'Class label        : {label} = "{train_ds.classes[label]}"')
print(f'class_to_idx       : {train_ds.class_to_idx}')


## 4. DataLoaders


In [ ]:
from ml.dataset import create_dataloaders

# Standard DataLoaders (shuffle training)
train_loader, val_loader, test_loader = create_dataloaders(
    batch_size=32,
    num_workers=2,          # 2 workers in Colab
    use_weighted_sampler=False
)

# Inspect one batch
batch_imgs, batch_labels = next(iter(train_loader))
print(f'Batch images shape : {batch_imgs.shape}')    # [32, 3, 224, 224]
print(f'Batch labels shape : {batch_labels.shape}')  # [32]
print(f'Batch labels dtype : {batch_labels.dtype}')  # int64
print(f'Unique classes in batch: {batch_labels.unique().tolist()}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ml.preprocessing import get_inverse_normalize

inv_norm = get_inverse_normalize()

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Training Batch (32 images, after augmentation)', fontsize=13, color='#e0e0e0')

for i, ax in enumerate(axes.flat):
    if i >= batch_imgs.size(0):
        ax.axis('off'); continue
    img_np = inv_norm(batch_imgs[i]).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    cls = train_ds.classes[batch_labels[i].item()]
    ax.set_title(cls[:5], fontsize=7, color='#e0e0e0', pad=2)
    ax.axis('off')
    ax.set_facecolor('#16213e')

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/results/eda/sample_batch.png', dpi=100, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
from ml.dataset import compute_class_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_weights = compute_class_weights(train_ds, device=device)

print(f'Class weights tensor: {class_weights}')
print(f'Device: {class_weights.device}')
print(f'dtype : {class_weights.dtype}')
print()
print(f'Highest weight → "{train_ds.classes[class_weights.argmax().item()]}" (index {class_weights.argmax().item()})')
print('This class will receive the largest gradient penalty during training.')


## 5. Automated Phase 2 Validation


In [ ]:
!python tests/validate_phase2.py


## ✅ Phase 2 Complete

**Deliverables:**
- `ml/preprocessing.py` — transform pipelines (train + val/test + inverse)
- `ml/dataset.py` — WasteDataset, DataLoaders, class weights, WeightedRandomSampler
- `ml/eda.py` — EDA charts and imbalance report
- `tests/validate_phase2.py` — 10-point validation suite
- `results/eda/` — all generated charts

**Key Findings:**
- TrashNet has 6 classes: cardboard(~403), glass(~501), metal(~410), paper(~594), plastic(~482), trash(~137)
- Severe class imbalance: paper is 4.3× larger than trash
- All images are originally ~512×384 px; resized to 224×224
- All images have 3 RGB channels
- Zero corrupted images found after Phase 1 verification
- No data leakage between splits

**Next: Phase 3 — Custom CNN implementation**
